# TE02_004 - Command Execution Accuracy KPI

This notebook validates the KPI described in `details.txt`: precise execution of commands with 100% accuracy compared to the official remote controller.

The notebook checks whether the required command/reference topics are present, compares command vectors exactly when available, and records an explicit `INCONCLUSIVE` result when the bag does not contain enough command data to validate the KPI.

## Validation Method

The primary validation compares the official remote-controller command topic against the firmware/executed command topic. For each reference command sample, the nearest executed command sample is selected within a configurable time window. The KPI passes only if every matched command vector is identical element-by-element.

The default topic mapping is configurable because the exact direction of `telejoy` and `telejoyout` can depend on the rosserial bridge configuration. If your bag uses the opposite direction, change `OFFICIAL_REMOTE_TOPIC` and `EXECUTED_COMMAND_TOPIC` in the configuration cell and rerun the notebook.

The notebook also extracts supporting evidence from `/statusjoy`, `/joints`, `/encoders`, and `/joint_states` so the report can show whether the bag contains controller status and machine-response data. These supporting signals do not replace the command-vector comparison required for a 100% command accuracy claim.

In [ ]:
from pathlib import Path
import json
import math
import os

import numpy as np
import pandas as pd

KPI_DIR = Path(str(os.environ.get('HOME')) + '/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004')
TE02_001_BAG_DIR = Path(str(os.environ.get('HOME')) + '/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_001/result/WBag/bag_20260707_121849')
BAG_DIR = Path(os.environ.get('TE02_004_BAG_DIR', TE02_001_BAG_DIR))

OUTPUT_DIR = KPI_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = OUTPUT_DIR / 'te02_004_command_accuracy_summary.csv'
MATCHED_COMMAND_CSV = OUTPUT_DIR / 'te02_004_matched_commands.csv'
TOPIC_COUNTS_CSV = OUTPUT_DIR / 'te02_004_topic_counts.csv'
SUPPORTING_SIGNALS_CSV = OUTPUT_DIR / 'te02_004_supporting_signals.csv'

# Change these two topics if your bag records the official remote and executed command in the opposite direction.
OFFICIAL_REMOTE_TOPIC = '/telejoy'
EXECUTED_COMMAND_TOPIC = '/telejoyout'

STATUS_TOPIC = '/statusjoy'
JOINTS_TOPIC = '/joints'
ENCODERS_TOPIC = '/encoders'
JOINT_STATES_TOPIC = '/joint_states'

COMMAND_VECTOR_LENGTH = 6
COMMAND_MATCH_WINDOW_MS = 100.0
REQUIRE_EXACT_COMMAND_MATCH = True

print(f'KPI output directory: {OUTPUT_DIR}')
print(f'Bag directory: {BAG_DIR}')

KPI output directory: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004
Bag directory: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_001/result/WBag/bag_20260707_121849


In [2]:
def load_metadata(bag_dir: Path):
    metadata_path = bag_dir / 'metadata.yaml'
    if not metadata_path.exists():
        raise FileNotFoundError(f'Missing rosbag metadata: {metadata_path}')
    try:
        import yaml
    except ImportError as exc:
        raise ImportError('Install PyYAML or run inside a ROS 2 environment with yaml available.') from exc
    with metadata_path.open('r', encoding='utf-8') as f:
        return yaml.safe_load(f)

metadata = load_metadata(BAG_DIR)
bag_info = metadata['rosbag2_bagfile_information']
storage_id = bag_info.get('storage_identifier', 'mcap')
duration_s = bag_info['duration']['nanoseconds'] / 1e9
topic_rows = []
for item in bag_info.get('topics_with_message_count', []):
    topic_meta = item['topic_metadata']
    topic_rows.append({
        'topic': topic_meta['name'],
        'type': topic_meta['type'],
        'message_count': int(item.get('message_count', 0)),
    })

topics_df = pd.DataFrame(topic_rows).sort_values('topic').reset_index(drop=True)
topics_df.to_csv(TOPIC_COUNTS_CSV, index=False)
print(f'Bag duration: {duration_s:.2f} s')
display(topics_df)

Bag duration: 161.72 s


,topic,type,message_count
0,/encoders,std_msgs/msg/Float32MultiArray,27081
1,/joint_states,sensor_msgs/msg/JointState,1616
2,/joints,std_msgs/msg/Float32MultiArray,1616
3,/statusjoy,std_msgs/msg/UInt8,27081
4,/telejoy,std_msgs/msg/Int8MultiArray,0
5,/tf,tf2_msgs/msg/TFMessage,1615
6,/tf_static,tf2_msgs/msg/TFMessage,28


In [3]:
def read_bag_topics(bag_dir: Path, selected_topics):
    try:
        import rosbag2_py
        from rclpy.serialization import deserialize_message
        from rosidl_runtime_py.utilities import get_message
    except ImportError as exc:
        raise ImportError(
            'This cell must be run in a ROS 2 Python environment with rosbag2_py, '
            'rclpy, and rosidl_runtime_py available.'
        ) from exc

    selected_topics = set(selected_topics)
    reader = rosbag2_py.SequentialReader()
    storage_options = rosbag2_py.StorageOptions(uri=str(bag_dir), storage_id=storage_id)
    converter_options = rosbag2_py.ConverterOptions('', '')
    reader.open(storage_options, converter_options)

    type_map = {t.name: t.type for t in reader.get_all_topics_and_types()}
    msg_type_cache = {topic: get_message(type_map[topic]) for topic in selected_topics if topic in type_map}

    rows = {topic: [] for topic in selected_topics}
    while reader.has_next():
        topic, data, timestamp_ns = reader.read_next()
        if topic not in selected_topics or topic not in msg_type_cache:
            continue
        msg = deserialize_message(data, msg_type_cache[topic])
        row = {'time_ns': int(timestamp_ns), 'time_s': timestamp_ns / 1e9}

        if hasattr(msg, 'data'):
            value = msg.data
            if isinstance(value, (list, tuple)):
                row['data'] = list(value)
            elif hasattr(value, 'tolist'):
                row['data'] = value.tolist()
            else:
                row['data'] = value
        elif topic == JOINT_STATES_TOPIC:
            row['name'] = list(msg.name)
            row['position'] = list(msg.position)
            row['velocity'] = list(msg.velocity)
            row['effort'] = list(msg.effort)
        rows[topic].append(row)

    return {topic: pd.DataFrame(topic_rows) for topic, topic_rows in rows.items()}

selected_topics = [
    OFFICIAL_REMOTE_TOPIC,
    EXECUTED_COMMAND_TOPIC,
    STATUS_TOPIC,
    JOINTS_TOPIC,
    ENCODERS_TOPIC,
    JOINT_STATES_TOPIC,
]
bag_frames = read_bag_topics(BAG_DIR, selected_topics)
for topic, df in bag_frames.items():
    print(f'{topic}: {len(df)} messages')

[ERROR] [1783434037.422673250] [rosbag2_storage]: Could not open '/home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_001/result/WBag/bag_20260707_121849/bag_20260707_121849_0.mcap.zstd' with 'mcap'. Error: invalid magic bytes in Header: 0x28B52FFD00586CD0
[ERROR] [1783434037.422743022] [rosbag2_storage]: Could not load/open plugin with storage id: 'mcap'
[INFO] [1783434037.422749574] [rosbag2_storage]: Available storage plugins: 'sqlite3', 'mcap', 


RuntimeError: No storage could be initialized. Abort

In [ ]:
def normalise_command_df(df: pd.DataFrame, topic_name: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['time_ns', 'time_s', 'topic', 'command', 'command_len', 'valid_shape'])
    out = df[['time_ns', 'time_s', 'data']].copy()
    out['topic'] = topic_name
    out['command'] = out['data'].apply(lambda x: list(x) if isinstance(x, (list, tuple)) else [x])
    out['command_len'] = out['command'].apply(len)
    out['valid_shape'] = out['command'].apply(
        lambda values: len(values) == COMMAND_VECTOR_LENGTH and all(-128 <= int(v) <= 127 for v in values)
    )
    return out.drop(columns=['data'])

def command_key(values):
    return tuple(int(v) for v in values)

official_df = normalise_command_df(bag_frames.get(OFFICIAL_REMOTE_TOPIC, pd.DataFrame()), OFFICIAL_REMOTE_TOPIC)
executed_df = normalise_command_df(bag_frames.get(EXECUTED_COMMAND_TOPIC, pd.DataFrame()), EXECUTED_COMMAND_TOPIC)

print(f'Official remote samples on {OFFICIAL_REMOTE_TOPIC}: {len(official_df)}')
print(f'Executed command samples on {EXECUTED_COMMAND_TOPIC}: {len(executed_df)}')
if not official_df.empty:
    display(official_df.head())
if not executed_df.empty:
    display(executed_df.head())

In [ ]:
def match_and_validate_commands(reference_df: pd.DataFrame, executed_df: pd.DataFrame) -> pd.DataFrame:
    if reference_df.empty or executed_df.empty:
        return pd.DataFrame(columns=[
            'reference_time_s', 'executed_time_s', 'dt_ms', 'reference_command',
            'executed_command', 'exact_match', 'valid_shape'
        ])

    ref = reference_df.sort_values('time_ns').copy()
    exe = executed_df.sort_values('time_ns').copy()
    ref['reference_time_ns'] = ref['time_ns']
    exe['executed_time_ns'] = exe['time_ns']

    matched = pd.merge_asof(
        ref,
        exe,
        on='time_ns',
        direction='nearest',
        tolerance=int(COMMAND_MATCH_WINDOW_MS * 1e6),
        suffixes=('_reference', '_executed'),
    )

    rows = []
    for _, row in matched.iterrows():
        reference_command = row.get('command_reference')
        executed_command = row.get('command_executed')
        has_match = isinstance(executed_command, list)
        dt_ms = np.nan
        if has_match and not pd.isna(row.get('executed_time_ns')):
            dt_ms = (int(row['executed_time_ns']) - int(row['reference_time_ns'])) / 1e6
        exact_match = bool(has_match and command_key(reference_command) == command_key(executed_command))
        valid_shape = bool(row.get('valid_shape_reference', False) and row.get('valid_shape_executed', False))
        rows.append({
            'reference_time_s': row.get('time_s_reference'),
            'executed_time_s': row.get('time_s_executed'),
            'dt_ms': dt_ms,
            'reference_command': reference_command,
            'executed_command': executed_command if has_match else None,
            'exact_match': exact_match,
            'valid_shape': valid_shape,
        })
    return pd.DataFrame(rows)

matched_commands = match_and_validate_commands(official_df, executed_df)
matched_commands.to_csv(MATCHED_COMMAND_CSV, index=False)
display(matched_commands.head(20))

matched_count = int(len(matched_commands))
exact_matches = int(matched_commands['exact_match'].sum()) if matched_count else 0
valid_shape_count = int(matched_commands['valid_shape'].sum()) if matched_count else 0
command_accuracy_pct = 100.0 * exact_matches / matched_count if matched_count else np.nan
print(f'Matched command samples: {matched_count}')
print(f'Exact command matches: {exact_matches}')
print(f'Command accuracy: {command_accuracy_pct if not np.isnan(command_accuracy_pct) else None}')

In [ ]:
supporting_rows = []
for topic in [STATUS_TOPIC, JOINTS_TOPIC, ENCODERS_TOPIC, JOINT_STATES_TOPIC]:
    df = bag_frames.get(topic, pd.DataFrame())
    row = {'topic': topic, 'samples': int(len(df)), 'available': bool(len(df) > 0)}
    if len(df) > 1:
        duration = (df['time_ns'].max() - df['time_ns'].min()) / 1e9
        row['duration_s'] = duration
        row['rate_hz'] = (len(df) - 1) / duration if duration > 0 else np.nan
    else:
        row['duration_s'] = np.nan
        row['rate_hz'] = np.nan
    if topic == STATUS_TOPIC and len(df) > 0:
        row['unique_values'] = sorted(set(int(v) for v in df['data'].tolist()))
    supporting_rows.append(row)

supporting_df = pd.DataFrame(supporting_rows)
supporting_df.to_csv(SUPPORTING_SIGNALS_CSV, index=False)
display(supporting_df)

In [ ]:
def topic_count(topic_name: str) -> int:
    match = topics_df.loc[topics_df['topic'] == topic_name, 'message_count']
    return int(match.iloc[0]) if len(match) else 0

summary_rows = []
official_count = topic_count(OFFICIAL_REMOTE_TOPIC)
executed_count = topic_count(EXECUTED_COMMAND_TOPIC)

if official_count == 0 or executed_count == 0:
    command_status = 'INCONCLUSIVE'
    command_reason = (
        f'Cannot validate 100% command accuracy because {OFFICIAL_REMOTE_TOPIC} has '
        f'{official_count} messages and {EXECUTED_COMMAND_TOPIC} has {executed_count} messages.'
    )
elif matched_count == 0:
    command_status = 'FAIL'
    command_reason = f'No command pairs matched within {COMMAND_MATCH_WINDOW_MS:.1f} ms.'
elif exact_matches == matched_count and valid_shape_count == matched_count:
    command_status = 'PASS'
    command_reason = 'Every matched command vector is valid and exactly equal to the official remote-controller vector.'
else:
    command_status = 'FAIL'
    command_reason = (
        f'{exact_matches}/{matched_count} matched command vectors are exactly equal; '
        f'{valid_shape_count}/{matched_count} command pairs have the expected vector shape.'
    )

summary_rows.append({
    'metric': 'command_vector_accuracy',
    'official_topic': OFFICIAL_REMOTE_TOPIC,
    'executed_topic': EXECUTED_COMMAND_TOPIC,
    'samples': matched_count,
    'accuracy_pct': command_accuracy_pct,
    'status': command_status,
    'reason': command_reason,
})

for _, row in supporting_df.iterrows():
    summary_rows.append({
        'metric': f'supporting_signal:{row["topic"]}',
        'official_topic': '',
        'executed_topic': '',
        'samples': int(row['samples']),
        'accuracy_pct': np.nan,
        'status': 'AVAILABLE' if row['available'] else 'MISSING',
        'reason': f'rate_hz={row.get("rate_hz")}',
    })

overall_status = command_status
summary_rows.append({
    'metric': 'TE02_004_overall',
    'official_topic': OFFICIAL_REMOTE_TOPIC,
    'executed_topic': EXECUTED_COMMAND_TOPIC,
    'samples': matched_count,
    'accuracy_pct': command_accuracy_pct,
    'status': overall_status,
    'reason': command_reason,
})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
display(summary_df)
print(f'Wrote summary: {SUMMARY_CSV}')
print(f'Overall TE02_004 status: {overall_status}')

## Interpretation

Use the generated `te02_004_command_accuracy_summary.csv` as the KPI evidence table.

If the overall result is `PASS`, the bag contains both command streams and every matched command vector is exactly equal to the official remote-controller vector.

If the overall result is `INCONCLUSIVE`, the bag does not contain enough command messages for a 100% command-accuracy claim. In that case, acquire a new bag including both command streams, normally `/telejoy` and `/telejoyout`, together with `/statusjoy`, `/joints`, `/encoders`, and `/joint_states`.

If the overall result is `FAIL`, inspect `te02_004_matched_commands.csv` to identify the exact timestamps and command vectors that differ from the official remote-controller reference.